# Stage 0 — Segmenter Exploration

Step-by-step walkthrough of `pipeline/stage0_segmenter.py`.
Run each cell and inspect the output before moving to the next step.

**Steps covered:**
1. PDF extraction (raw text + per-page text)
2. Speaker turn detection
3. Role classification
4. Q&A pairing
5. Long-turn splitting
6. Page attribution + final Chunk objects

## Setup

In [ ]:
import sys
sys.path.insert(0, ".")

import re
from pypdf import PdfReader
from pipeline.stage0_segmenter import (
    _find_speaker_matches,
    _boundary_kind,
    _split_into_turns,
    _extract_management_roster,
    _classify_roles,
    _create_qa_pairs,
    _split_chunk,
    _build_page_boundaries,
    _char_to_page,
    extract_pages_from_pdf,
)
from pipeline.models import ChunkRole

# ── Pick transcript ──────────────────────────────────────────────────────────
# Change this to any transcript in the transcripts/ folder
PDF_PATH = "transcripts/fineotex_chemical_Q4_FY26.pdf"
# PDF_PATH = "transcripts/asian_paints_Q4_FY26.pdf"
# PDF_PATH = "transcripts/sandhar_technologies_Q4_FY26.pdf"
# PDF_PATH = "transcripts/mold-tek_packaging_Q4_FY26.pdf"

print(f"Using: {PDF_PATH}")

---
## Step 1 — PDF Extraction

Extract full text and per-page text from the PDF using `pypdf`.

**What to check:**
- Does the raw text look sensible?
- Are there header/footer artifacts injected mid-sentence? (Company name, date, Page N of M)
- Does page count match the PDF?

In [ ]:
transcript_pages = extract_pages_from_pdf(PDF_PATH)
transcript_text = "".join(transcript_pages[pn] for pn in sorted(transcript_pages))

print(f"Pages extracted : {len(transcript_pages)}")
print(f"Total characters: {len(transcript_text):,}")
print(f"Total words     : {len(transcript_text.split()):,}")

In [ ]:
# Inspect the first 3000 characters — check for participant list and any artifacts
print(transcript_text[:3000])

In [ ]:
# Inspect a mid-document page to spot header/footer injection
# Change page number to a middle page of the transcript
PAGE_TO_INSPECT = 3
print(f"--- Page {PAGE_TO_INSPECT} ---")
print(transcript_pages.get(PAGE_TO_INSPECT, "Page not found"))

---
## Step 2 — Speaker Turn Detection

`_find_speaker_matches` finds all candidate speaker headers using 5 regex patterns (D > E > A > B > C priority).

`_split_into_turns` filters to valid boundaries (strong = after newline, weak = after punctuation but only if speaker recurs) and splits the text between them.

**What to check:**
- Are all expected speakers detected?
- Any false positives (e.g. `Outlook:` or `Note:` matched as a speaker)?
- Any real speakers missed entirely?

In [ ]:
# All raw regex matches before boundary filtering
all_matches = _find_speaker_matches(transcript_text)
print(f"Total speaker header matches: {len(all_matches)}")
print()
print(f"{'Pattern':<10} {'Name':<35} {'Position'}")
print("-" * 65)
for start, end, name, pid in all_matches[:40]:  # first 40
    print(f"  {pid:<8} {name:<35} char {start}")

In [ ]:
# Boundary classification for each match
print(f"{'Boundary':<10} {'Pattern':<10} {'Name'}")
print("-" * 55)
for start, end, name, pid in all_matches[:40]:
    kind = _boundary_kind(transcript_text, start)
    print(f"  {str(kind):<8} {pid:<10} {name}")

In [ ]:
# Final turns after boundary filtering + recurrence check
raw_turns = _split_into_turns(transcript_text)
print(f"Raw turns detected: {len(raw_turns)}")
print()

# Unique speakers
from collections import Counter
speaker_counts = Counter(t[0] for t in raw_turns)
print("Speaker turn counts:")
for speaker, count in speaker_counts.most_common():
    print(f"  {count:>3}x  {speaker}")

In [ ]:
# Inspect specific turns — change index to check any turn
TURN_INDEX = 0

speaker, text, cs, ce = raw_turns[TURN_INDEX]
print(f"Turn {TURN_INDEX}: [{speaker}]")
print(f"Chars {cs}–{ce} | Words: {len(text.split())}")
print()
print(text[:1000])

---
## Step 3 — Role Classification

`_extract_management_roster` parses the `MANAGEMENT: ... MODERATOR: ...` participant list at the top of the transcript. This is the authoritative source — doesn't depend on when a speaker first talks.

`_classify_roles` assigns `MANAGEMENT / ANALYST / MODERATOR` to each turn.

**What to check:**
- Is the management roster correctly parsed?
- Are analyst names correctly classified (not misclassified as management)?
- Did any management speaker get classified as analyst (e.g. a CFO who only speaks in Q&A)?

In [ ]:
mgmt_roster = _extract_management_roster(transcript_text)
print(f"Management roster ({len(mgmt_roster)} speakers):")
for name in sorted(mgmt_roster):
    print(f"  {name}")

if not mgmt_roster:
    print("WARNING: No roster found — falling back to first-4-turns heuristic")

In [ ]:
turns_with_roles = _classify_roles(raw_turns, mgmt_roster)

role_counts = Counter(t[4] for t in turns_with_roles)
print("Role breakdown:")
for role, count in role_counts.most_common():
    print(f"  {role.value:<12} {count} turns")

print()
print(f"{'Role':<14} {'Speaker':<35} {'Words'}")
print("-" * 60)
for speaker, text, cs, ce, role in turns_with_roles:
    print(f"  {role.value:<12} {speaker:<35} {len(text.split())}")

---
## Step 4 — Q&A Pairing

`_create_qa_pairs` combines each analyst question with the following management answer into a single chunk.

Rules:
- `ANALYST` → scan forward (skip `MODERATOR`) → find `MANAGEMENT` → merge
- `MODERATOR` with `?` → treated as analyst question (written submission)
- Consecutive `MANAGEMENT` turns after an answer → absorbed (MD + CFO both answer)
- Solo `MANAGEMENT` turn → standalone chunk (`is_qa_pair=False`)
- Standalone `MODERATOR` → dropped

**What to check:**
- Are Q&A pairs correctly formed?
- Any analyst question dropped (no following management turn)?
- Does the combined text look right?

In [ ]:
chunk_dicts = _create_qa_pairs(turns_with_roles)

qa_pairs = sum(1 for c in chunk_dicts if c["is_qa_pair"])
solo = len(chunk_dicts) - qa_pairs
print(f"Total chunks  : {len(chunk_dicts)}")
print(f"Q&A pairs     : {qa_pairs}")
print(f"Solo mgmt     : {solo}")

In [ ]:
# Overview of all chunks
print(f"{'#':<5} {'QA?':<6} {'Speaker':<30} {'Words'}")
print("-" * 55)
for i, c in enumerate(chunk_dicts):
    qa = "yes" if c["is_qa_pair"] else "no"
    print(f"  {i:<3} {qa:<6} {c['speaker']:<30} {len(c['text'].split())}")

In [ ]:
# Inspect a specific chunk — change index
CHUNK_INDEX = 1

c = chunk_dicts[CHUNK_INDEX]
print(f"Chunk {CHUNK_INDEX}: [{c['speaker']}] | Q&A pair: {c['is_qa_pair']} | Words: {len(c['text'].split())}")
print()
print(c["text"][:2000])

---
## Step 5 — Long-Turn Splitting

Chunks over 1,500 words are split at `\n\n` paragraph boundaries with 200-char overlap between sub-chunks.

IDs become: `chunk_002` → `chunk_002a`, `chunk_002b`, ...

**What to check:**
- Which chunks were split?
- Does the overlap stitch sentences correctly across sub-chunks?

In [ ]:
flat = []
for cd in chunk_dicts:
    flat.extend(_split_chunk(cd))

split_chunks = [(cd, si) for cd, si in flat if si is not None]
print(f"Chunks before splitting : {len(chunk_dicts)}")
print(f"Chunks after splitting  : {len(flat)}")
print(f"Sub-chunks created      : {len(split_chunks)}")

if split_chunks:
    print()
    print("Chunks that were split:")
    seen = set()
    for cd, si in split_chunks:
        key = cd["speaker"]
        if key not in seen:
            seen.add(key)
            print(f"  [{cd['speaker']}] sub-chunk {si}, words: {len(cd['text'].split())}")

In [ ]:
# If any chunks were split, inspect the overlap between sub-chunk 0 and 1
if split_chunks:
    sub_0 = next((cd for cd, si in flat if si == 0), None)
    sub_1 = next((cd for cd, si in flat if si == 1), None)
    if sub_0 and sub_1:
        print("--- End of sub-chunk 0 (last 300 chars) ---")
        print(sub_0["text"][-300:])
        print()
        print("--- Start of sub-chunk 1 (first 300 chars) ---")
        print(sub_1["text"][:300])
else:
    print("No chunks were split — all under 1,500 words.")

---
## Step 6 — Page Attribution + Final Chunk Objects

`_build_page_boundaries` maps each page number to `(char_start, char_end)` in the concatenated text.

`_char_to_page` looks up each chunk's `char_start` to assign `page_start` and `page_end`.

**What to check:**
- Are page numbers plausible for each chunk?
- Do `chunk_id` labels look correct (sequential, `a/b/c` suffix for splits)?

In [ ]:
page_boundaries = _build_page_boundaries(transcript_pages)

print(f"Page boundaries built for {len(page_boundaries)} pages")
print()
print(f"{'Page':<8} {'char_start':>12} {'char_end':>12}")
print("-" * 35)
for pn in sorted(page_boundaries)[:10]:  # first 10 pages
    s, e = page_boundaries[pn]
    print(f"  {pn:<6} {s:>12,} {e:>12,}")

In [ ]:
from pipeline.models import Chunk

chunks = []
parent_num = 0

for chunk_dict, sub_idx in flat:
    if sub_idx is None or sub_idx == 0:
        parent_num += 1

    if sub_idx is None:
        chunk_id = f"chunk_{parent_num:03d}"
    else:
        chunk_id = f"chunk_{parent_num:03d}{chr(ord('a') + sub_idx)}"

    page_start = _char_to_page(chunk_dict["char_start"], page_boundaries)
    page_end = _char_to_page(
        max(chunk_dict["char_end"] - 1, chunk_dict["char_start"]),
        page_boundaries,
    )

    chunks.append(Chunk(
        chunk_id=chunk_id,
        speaker=chunk_dict["speaker"],
        role=chunk_dict["role"],
        page_start=page_start,
        page_end=page_end,
        text=chunk_dict["text"],
        char_start=chunk_dict["char_start"],
        char_end=chunk_dict["char_end"],
        is_qa_pair=chunk_dict["is_qa_pair"],
    ))

print(f"Final chunk count: {len(chunks)}")
print()
print(f"{'chunk_id':<14} {'Q&A?':<6} {'Pages':<10} {'Speaker':<30} {'Words'}")
print("-" * 75)
for c in chunks:
    qa = "yes" if c.is_qa_pair else "no"
    pages = f"p{c.page_start}" if c.page_start == c.page_end else f"p{c.page_start}–{c.page_end}"
    print(f"  {c.chunk_id:<12} {qa:<6} {pages:<10} {c.speaker:<30} {len(c.text.split())}")

In [ ]:
# Inspect any final chunk in full
FINAL_CHUNK_INDEX = 0

c = chunks[FINAL_CHUNK_INDEX]
print(f"chunk_id   : {c.chunk_id}")
print(f"speaker    : {c.speaker}")
print(f"role       : {c.role}")
print(f"pages      : {c.page_start}–{c.page_end}")
print(f"is_qa_pair : {c.is_qa_pair}")
print(f"words      : {len(c.text.split())}")
print()
print(c.text)

---
## Summary

Run the full `segment()` pipeline in one call and compare against the step-by-step output above.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from pipeline.stage0_segmenter import segment

result = segment(transcript_text, transcript_pages)

print(f"\nTotal chunks  : {len(result)}")
print(f"Q&A pairs     : {sum(1 for c in result if c.is_qa_pair)}")
print(f"Solo mgmt     : {sum(1 for c in result if not c.is_qa_pair)}")